# Embedding + ChromaDB indexing — a diagnostic client of `engineering_rag`

**This notebook contains no embedding or storage implementation.** It imports the
production package (`engineering_rag.services.embedder`, `engineering_rag.databases.chroma`,
`engineering_rag.pipelines.indexing_pipeline`) and calls its public API — exactly like
`01_docling_exploration.ipynb` does for the parser and chunker.

Runs top-to-bottom, uses repository-relative paths, no hidden state. The BGE model download
happens once (Hugging Face cache) in the cell that loads it — clearly marked below.

In [1]:
import json
import os

from engineering_rag.utils.paths import repo_root

ROOT = repo_root()
os.chdir(
    ROOT
)  # so relative paths in YAML profiles resolve against the repo root, matching the CLI (always invoked from repo root)
print("Repository root:", ROOT)

Repository root: E:\engineering-rag-parser


## 1. Load `chunks.jsonl` from a BGE-aligned chunker run

In [2]:
CHUNK_RUN_DIR = ROOT / "data/output/chunker/Instrumentation-and-Control-Engineering/20260825T073605Z-01e4d6fa"
chunks_path = CHUNK_RUN_DIR / "chunks.jsonl"

records = [json.loads(line) for line in chunks_path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"{len(records)} chunk(s) loaded from {chunks_path.relative_to(ROOT)}")

113 chunk(s) loaded from data\output\chunker\Instrumentation-and-Control-Engineering\20260825T073605Z-01e4d6fa\chunks.jsonl


## 2. Inspect `retrieval_text` and metadata

In [3]:
sample = records[10]
print("chunk_id:", sample["chunk_id"])
print("content_type:", sample["content_type"])
print("heading_path:", sample["heading_path"])
print("token_count (chunker's own tokenizer):", sample["token_count"])
print()
print("retrieval_text:")
print(sample["retrieval_text"][:500])

chunk_id: chunk_f21ad2aeccdc2e91
content_type: text
heading_path: ['Section 4: Categorized C&I Deliverables: Control System and Logic', 'Input/Output (I/O) Lists and Schedules']
token_count (chunker's own tokenizer): 27

retrieval_text:
Section 4: Categorized C&I Deliverables: Control System and Logic > Input/Output (I/O) Lists and Schedules
Purpose and Development

www.instrunexus.com                                                           Page 1 of 27

Key Content

### Control System Architecture Drawings and Specifications


## 3. Load `BAAI/bge-base-en-v1.5`

**Model download / cache behavior:** the first run of this cell downloads the model through
the normal Hugging Face cache (`~/.cache/huggingface`, or `HF_HOME` if set) — nothing is
written into the repository. Subsequent runs reuse the cache and load instantly offline.

In [4]:
from engineering_rag.services.embedder.bge import BGEEmbeddingService
from engineering_rag.services.embedder.config import EmbedderConfig

embedder = BGEEmbeddingService(EmbedderConfig())
info = embedder.model_info()
print("model_name:", info.model_name)
print("resolved_revision:", info.resolved_revision)
print("dimension:", info.dimension)
print("max_seq_length:", info.max_seq_length)
print("device:", info.device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model_name: BAAI/bge-base-en-v1.5
resolved_revision: a5beb1e3e68b9ab74eb54cfd186867f64f240e1a
dimension: 768
max_seq_length: 512
device: cpu


## 4. Embed a passage (no prefix — `retrieval_text` embedded directly)

In [5]:
passage_records, batch_stats = embedder.embed_passages([sample["chunk_id"]], [sample["retrieval_text"]])
passage_vector = passage_records[0].vector
print("vector length:", len(passage_vector))
print("first 8 values:", [round(v, 4) for v in passage_vector[:8]])
print("duration_s:", batch_stats.duration_s, " vec/s:", batch_stats.vectors_per_second)

vector length: 768
first 8 values: [-0.0309, 0.023, -0.0319, 0.0041, 0.0687, 0.0076, -0.0021, 0.0333]
duration_s: 0.1251  vec/s: 7.99


## 5. Embed a query (required BGE instruction prefix)

Queries get the exact prefix `"Represent this sentence for searching relevant passages: "`;
passages never do — see `docs/indexing/MENTOR_EXPLANATION.md` for why this asymmetry matters.

In [6]:
query_vector = embedder.embed_query("What is a control system?")
print("vector length:", len(query_vector))
print("first 8 values:", [round(v, 4) for v in query_vector[:8]])

vector length: 768
first 8 values: [-0.0585, 0.0415, 0.0018, 0.0245, 0.0542, -0.0126, 0.0115, -0.0041]


## 6. Verify 768 dimensions

In [7]:
assert len(passage_vector) == 768
assert len(query_vector) == 768
print("Both vectors are 768-dimensional.")

Both vectors are 768-dimensional.


## 7. Verify normalization (L2 norm ≈ 1.0)

In [8]:
import math


def l2_norm(vec):
    return math.sqrt(sum(v * v for v in vec))


print("passage L2 norm:", round(l2_norm(passage_vector), 6))
print("query L2 norm:  ", round(l2_norm(query_vector), 6))

passage L2 norm: 1.0
query L2 norm:   1.0


## 8. Create/open a Chroma collection

In [9]:
from engineering_rag.databases.chroma import CollectionIdentity, get_client, open_or_create_collection
from engineering_rag.databases.chroma.config import ChromaConfig

DEMO_CHROMA_PATH = ROOT / "data/output/databases/chroma_notebook_demo"
demo_config = ChromaConfig(persistence_path=DEMO_CHROMA_PATH, collection_name="notebook_demo_v1")

client = get_client(demo_config.persistence_path, telemetry=demo_config.telemetry)
identity = CollectionIdentity(
    model_name=info.model_name,
    embedding_dimension=info.dimension,
    distance_metric=demo_config.distance_metric,
    tokenizer_name=info.tokenizer_name,
    corpus_id="notebook-demo",
)
collection = open_or_create_collection(client, demo_config, identity)
print("Collection:", collection.name, " count before:", collection.count())

Collection: notebook_demo_v1  count before: 5


## 9. Store a small deterministic sample

In [10]:
from engineering_rag.databases.chroma import chroma_safe_metadata, content_hash, ingest_batch

demo_sample = records[:5]
demo_ids = [r["chunk_id"] for r in demo_sample]
demo_texts = [r["retrieval_text"] for r in demo_sample]
demo_records, _ = embedder.embed_passages(demo_ids, demo_texts)
demo_vectors = [r.vector for r in demo_records]

demo_metadatas = []
for r in demo_sample:
    fields = {
        "document_id": r["document_id"],
        "content_type": r["content_type"],
        "section_title": r.get("section_title"),
        "chunk_run_id": CHUNK_RUN_DIR.name,
    }
    safe = chroma_safe_metadata(fields)
    safe["content_hash"] = content_hash(r["retrieval_text"], safe)
    demo_metadatas.append(safe)

outcome = ingest_batch(
    collection,
    ids=demo_ids,
    embeddings=demo_vectors,
    documents=demo_texts,
    metadatas=demo_metadatas,
    idempotent=demo_config.idempotent,
)
print("inserted:", outcome.inserted_ids)
print("existing_identical:", outcome.existing_identical_ids)
print("collection count after:", collection.count())

inserted: []
existing_identical: ['chunk_12853b0e951df7d3', 'chunk_43856312430296c3', 'chunk_359f3f41bfa09184', 'chunk_ab16bedfbf6f078b', 'chunk_8b37126afe96d3a0']
collection count after: 5


## 10. Reopen the persistent collection (fresh client)

In [11]:
fresh_client = get_client(demo_config.persistence_path, telemetry=demo_config.telemetry)
reopened = fresh_client.get_collection(name=demo_config.collection_name)
print("Reopened collection count:", reopened.count())

Reopened collection count: 5


## 11. Run a diagnostic query

In [12]:
result = reopened.query(
    query_embeddings=[query_vector],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)
for cid, dist, doc in zip(result["ids"][0], result["distances"][0], result["documents"][0], strict=True):
    print(f"{cid}  distance={dist:.4f}")
    print("  ", doc[:100].replace(chr(10), " "))

chunk_359f3f41bfa09184  distance=0.5490
   Phase 2: Front-End Engineering Design (FEED) and Feasibility Activity: Process Blueprints and Contro
chunk_12853b0e951df7d3  distance=0.5809
   Section 1: Foundational Framework of Instrumentation Design Engineering (IDE)  1.1 Role, Mandate, an
chunk_43856312430296c3  distance=0.6099
   Project Lifecycle Phasing and C&I Scope Evolution Section 2: The Instrumentation Design Engineering 


## 12. Inspect IDs, documents and metadata

In [13]:
fetched = reopened.get(ids=demo_ids, include=["documents", "metadatas"])
for cid, doc, meta in zip(fetched["ids"], fetched["documents"], fetched["metadatas"], strict=True):
    print(cid, "->", meta.get("content_type"), "|", doc[:60].replace(chr(10), " "))

chunk_12853b0e951df7d3 -> text | Section 1: Foundational Framework of Instrumentation Design 
chunk_43856312430296c3 -> text | Project Lifecycle Phasing and C&I Scope Evolution Section 2:
chunk_359f3f41bfa09184 -> text | Phase 2: Front-End Engineering Design (FEED) and Feasibility
chunk_ab16bedfbf6f078b -> text | Phase 3: Detailed Engineering Design (Execution Core) Activi
chunk_8b37126afe96d3a0 -> text | Phase 4: Procurement, Installation, and Integration 2.5 Phas


## 13. Run the production indexing pipeline

This is the same code path `engrag-index build` uses — no logic is duplicated here, this cell
just calls the pipeline function directly against the *real* production collection
(`configs/indexing_production.yaml`), not the notebook's demo collection above.

In [14]:
from engineering_rag.pipelines.indexing_config import load_indexing_config
from engineering_rag.pipelines.indexing_pipeline import run_indexing_pipeline

production_config = load_indexing_config(ROOT / "configs/indexing_production.yaml")
result = run_indexing_pipeline(CHUNK_RUN_DIR, production_config)
print("status:", result.status)
print("run_dir:", result.run_dir)
print("chunk_count:", result.chunk_count)
print("collection:", result.collection_name, "at", result.chroma_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

status: PASS
run_dir: data\output\indexing\engineering_documents_v1\20260825T083244Z-4eeb2768
chunk_count: 113
collection: engineering_documents_v1 at data\output\databases\chroma


## 14. Read the validation report

In [15]:
validation_report_path = result.run_dir / "index_validation_report.json"
report = json.loads(validation_report_path.read_text(encoding="utf-8"))
print("status:", report["status"])
for check in report["checks"]:
    mark = "PASS" if check["passed"] else "FAIL"
    print(f"  [{mark}] {check['check_id']} (severity={check['severity']}, gate={check['gate']})")
print()
print("human_review_items:", report["human_review_items"])

status: PASS
  [PASS] chunks_schema_supported (severity=CRITICAL, gate=True)
  [PASS] chunker_run_passed_validation (severity=CRITICAL, gate=True)
  [PASS] tokenizer_family_match (severity=CRITICAL, gate=True)
  [PASS] no_silent_truncation (severity=CRITICAL, gate=True)
  [PASS] all_expected_ids_present (severity=CRITICAL, gate=True)
  [PASS] no_duplicate_or_unexpected_ids (severity=CRITICAL, gate=True)
  [PASS] collection_count_covers_input (severity=CRITICAL, gate=True)
  [PASS] vectors_valid (severity=CRITICAL, gate=True)
  [PASS] cosine_distance_metric (severity=CRITICAL, gate=True)
  [PASS] round_trip_storage_matches (severity=CRITICAL, gate=True)
  [PASS] self_retrieval_rank_one (severity=WARNING, gate=False)
  [PASS] relative_paths_portable (severity=WARNING, gate=False)

human_review_items: []


## Summary

- Loaded real chunker output (`chunks.jsonl`) from a BGE-aligned chunk run.
- Loaded `BAAI/bge-base-en-v1.5`, embedded a passage (no prefix) and a query (required prefix),
  confirmed 768 dimensions and L2-normalization on both.
- Created a Chroma collection, stored a small sample, reopened it in a fresh client, queried
  and inspected it.
- Ran the actual production indexing pipeline (`run_indexing_pipeline`) against the real
  engineering-document chunk run and read back its validation report.

See `docs/indexing/` for the full architecture, configuration, validation-gate list, and the
milestone completion report with real-corpus evidence.